# LeetCode 4041 — Minimum Operations to Form Subset Sum II bằng Python

- Link: https://leetcode.com/problems/minimum-operations-to-form-subset-sum-ii/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động (DP), Knapsack 0-1, Knapsack theo nhóm, Toán (nhân/chia 2)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minOperations(self, nums: list[int], sum: int) -> int:
```

## 1. Giới thiệu & mục tiêu học

Bạn có một mảng `nums`. Mỗi phép biến đổi cho phép lấy một phần tử đang có giá trị `x` và đổi nó thành `2 * x` hoặc `floor(x / 2)`. Nhân và chia có thể **đan xen tuỳ ý**. Hỏi số phép biến đổi **ít nhất** để sau khi biến đổi, tồn tại một **tập con** của mảng có tổng đúng bằng `sum` (không được thì trả `-1`).

Bài này ghép hai ý quen thuộc: (1) "mỗi phần tử có nhiều **phiên bản** với giá trị và chi phí khác nhau" và (2) bài toán cái túi 0-1 (chọn hoặc không chọn mỗi phần tử). Phần khó nhất không phải DP mà là **nhìn ra dạng chuẩn** của chuỗi thao tác: mọi chuỗi nhân/chia đều rút gọn được thành "chia `i` lần rồi nhân `j` lần".

Sau notebook này bạn sẽ:

1. Biến một chuỗi thao tác tuỳ ý thành **dạng chuẩn** `floor(x / 2^i) * 2^j` với chi phí `i + j`.
2. Hiểu bài toán cái túi 0-1 và vì sao phải **duyệt dung lượng từ lớn về nhỏ** khi dùng mảng 1 chiều.
3. Mở rộng sang **knapsack theo nhóm**: mỗi phần tử là một nhóm các lựa chọn (value, cost), chọn nhiều nhất một lựa chọn trong nhóm.
4. Viết 4 phiên bản: vét cạn bằng BFS, DP 2 chiều, DP 1 chiều, DP top-down (đệ quy nhớ), rồi đối chiếu chúng bằng test ngẫu nhiên.
5. Phân tích độ phức tạp `O(n · S · log M · log S)` và biết khi nào cách làm này đủ nhanh.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho mảng số nguyên dương `nums` và số nguyên `sum`.

- Một **phép biến đổi**: chọn một phần tử đang có giá trị `x`, thay nó bằng `2 * x` hoặc `x // 2` (chia nguyên, làm tròn xuống).
- Với **mỗi phần tử**, bạn được áp dụng bao nhiêu phép biến đổi tuỳ ý, thứ tự nhân/chia tuỳ ý.
- Mục tiêu: sau khi biến đổi, có một **tập con** của mảng mới (có thể chỉ gồm một phần tử, mỗi phần tử dùng tối đa một lần) có tổng **đúng bằng** `sum`.
- Trả về số phép biến đổi **tối thiểu**, hoặc `-1` nếu không thể.

**Ví dụ 1:** `nums = [10, 2]`, `sum = 13` → `3`.
Chia `10 → 5` (1 phép), nhân `2 → 4 → 8` (2 phép). Mảng thành `[5, 8]`, tập `{5, 8}` có tổng 13.

**Ví dụ 2:** `nums = [6, 3]`, `sum = 8` → `2`.
Biến `3` thành `2` bằng 2 phép: `3 → 1 → 2`. Mảng `[6, 2]`, tập `{6, 2}` có tổng 8. Đây chính là chỗ nhân/chia **đan xen** hữu ích.

**Ví dụ 3:** `nums = [2, 2]`, `sum = 7` → `-1`.

**Ràng buộc**

- `1 <= nums.length <= 100`
- `1 <= nums[i] <= 500`
- `1 <= sum <= 5000`

Lưu ý: tên tham số là `sum` (trùng tên hàm dựng sẵn `sum` của Python). Trong code nộp ta **không** gọi hàm `sum(...)` bên trong phương thức để tránh nhầm.

In [1]:
from typing import List

EXAMPLES = [
    ([10, 2], 13, 3),
    ([6, 3], 8, 2),
    ([2, 2], 7, -1),
]
for nums, target, expected in EXAMPLES:
    print(f"nums={nums}, sum={target}  ->  kỳ vọng {expected}")

nums=[10, 2], sum=13  ->  kỳ vọng 3
nums=[6, 3], sum=8  ->  kỳ vọng 2
nums=[2, 2], sum=7  ->  kỳ vọng -1


## 3. Kiến thức nền

Trước khi giải, ta ôn lại từng công cụ nhỏ sẽ dùng.

### 3.1. `//` (chia nguyên) và `* 2`

`x // 2` bỏ phần thập phân (làm tròn **xuống** với số dương) — đúng với `floor(x / 2)` trong đề. Còn `x / 2` cho số thực, **không** dùng.

In [2]:
for x in [10, 7, 3, 1]:
    print(f"x={x}:  x//2 = {x // 2},  x/2 = {x / 2},  x*2 = {x * 2}")

# Nhân rồi chia luôn quay lại số cũ, vì (2x)//2 == x
print(all((2 * x) // 2 == x for x in range(1, 1000)))
# Nhưng chia rồi nhân thì KHÔNG quay lại số cũ khi x lẻ
print((7 // 2) * 2)   # 6, không phải 7

x=10:  x//2 = 5,  x/2 = 5.0,  x*2 = 20
x=7:  x//2 = 3,  x/2 = 3.5,  x*2 = 14
x=3:  x//2 = 1,  x/2 = 1.5,  x*2 = 6
x=1:  x//2 = 0,  x/2 = 0.5,  x*2 = 2
True
6


### 3.2. Danh sách "chi phí nhỏ nhất để đạt tổng w" — mảng `f` và vô cực

Ta sẽ dùng mảng `f` dài `sum + 1`, trong đó `f[w]` = số phép biến đổi ít nhất để một tập con có tổng đúng `w`. Với `w` chưa đạt được ta ghi `INF` (một số rất lớn, ví dụ `10**9`) để phép `min(...)` luôn ưu tiên giá trị thật.

`f[0] = 0` vì tập rỗng có tổng 0 mà không tốn phép nào.

In [3]:
INF = 10**9
S = 5
f = [0] + [INF] * S
print(f)               # [0, 1000000000, ...]
f[3] = min(f[3], 2)    # đạt được tổng 3 với 2 phép
print(f)
print(min(INF, 7), min(INF + 1, INF))   # INF "thua" mọi giá trị thật

[0, 1000000000, 1000000000, 1000000000, 1000000000, 1000000000]
[0, 1000000000, 1000000000, 2, 1000000000, 1000000000]
7 1000000000


### 3.3. Bài toán cái túi 0-1 và hướng duyệt dung lượng

Cái túi 0-1: có các món, món `k` có `giá trị v_k` và `chi phí c_k`; mỗi món chọn tối đa **một lần**. Ở đây "giá trị" chính là phần đóng góp vào tổng, và ta muốn tổng **đúng bằng** `sum` với chi phí nhỏ nhất.

Với mảng 1 chiều `f`, khi xử lý một món ta cập nhật `f[w] = min(f[w], f[w - v] + c)`. Hướng duyệt `w` quyết định món đó có bị dùng **nhiều lần** hay không:

- Duyệt `w` **từ nhỏ lên lớn**: `f[w - v]` có thể đã được cập nhật bởi chính món này → món bị dùng lại (knapsack không giới hạn).
- Duyệt `w` **từ lớn về nhỏ**: `f[w - v]` (với `v >= 1`) vẫn là giá trị **trước** món này → món chỉ dùng một lần (knapsack 0-1).

In [4]:
def coin_min_count(coins, S, descending):
    """Số đồng tối thiểu để tạo đúng S; mỗi đồng chỉ nên dùng 1 lần."""
    INF = 10**9
    f = [0] + [INF] * S
    for v in coins:
        order = range(S, v - 1, -1) if descending else range(v, S + 1)
        for w in order:
            f[w] = min(f[w], f[w - v] + 1)
    return f[S]

# Chỉ có MỘT đồng 5, muốn tổng 10: thực tế không thể (cần hai đồng 5)
print("duyệt giảm (đúng 0-1):", coin_min_count([5], 10, True) >= 10**9)   # True: không tạo được
print("duyệt tăng (sai 0-1):  ", coin_min_count([5], 10, False))          # 2: dùng đồng 5 hai lần!

duyệt giảm (đúng 0-1): True
duyệt tăng (sai 0-1):   2


### 3.4. Knapsack theo nhóm (mỗi phần tử có nhiều "phiên bản")

Trong bài này, mỗi phần tử `x` có **nhiều phiên bản**: ví dụ `x = 6` có thể thành `6` (0 phép), `12` (1 phép), `3` (1 phép), `6` (2 phép: chia rồi nhân)... Nhưng trong tập con ta chỉ lấy **tối đa một phiên bản** của mỗi phần tử gốc (vì đó chỉ là một phần tử của mảng).

Đó là **knapsack theo nhóm**: mỗi nhóm có danh sách lựa chọn `(value, cost)`, chọn tối đa một lựa chọn mỗi nhóm. Cách làm chuẩn: với mỗi nhóm, tính mảng mới `g` từ mảng cũ `f` (cho mọi lựa chọn dùng `f` **cũ**), rồi gán `f = g`.

In [5]:
# Nhóm 1: chọn 1 trong {(value=2, cost=1), (value=3, cost=5)} hoặc bỏ qua
# Nhóm 2: chọn 1 trong {(value=3, cost=2)} hoặc bỏ qua
groups = [[(2, 1), (3, 5)], [(3, 2)]]
S = 6
INF = 10**9
f = [0] + [INF] * S
for g_opts in groups:
    g = f[:]                          # mặc định: bỏ qua nhóm này
    for v, c in g_opts:
        for w in range(v, S + 1):
            g[w] = min(g[w], f[w - v] + c)   # luôn đọc f CŨ
    f = g
    print(f)
print("tổng 5 cần", f[5], "phép")   # 2 (nhóm 1, chi phí 1) + 3 (nhóm 2, chi phí 2) = 3
print("tổng 6 cần", f[6], "phép")   # 3 (nhóm 1, chi phí 5) + 3 (nhóm 2, chi phí 2) = 7
print("tổng 4 cần", f[4] if f[4] < INF else "không thể")   # nhóm 1 chỉ cho 2 hoặc 3, nhóm 2 chỉ cho 3 -> không ra 4

[0, 1000000000, 1, 5, 1000000000, 1000000000, 1000000000]
[0, 1000000000, 1, 2, 1000000000, 3, 7]
tổng 5 cần 3 phép
tổng 6 cần 7 phép
tổng 4 cần không thể


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Mỗi phần tử có một "bảng giá"

Với phần tử `x`, ta có thể hỏi: "muốn biến `x` thành giá trị `y` thì tốn tối thiểu bao nhiêu phép?". Tập `y` đạt được và chi phí chính là **bảng giá** của `x`. Bài toán trở thành: với mỗi phần tử chọn *không dùng* hoặc *một dòng* trong bảng giá của nó, sao cho tổng giá trị đúng `sum` và tổng chi phí nhỏ nhất.

### 4.2. Tính tay ví dụ 1: `nums = [10, 2]`, `sum = 13`

- Bảng giá của `10` (giá trị ≤ 13): `10 (0 phép)`, `5 (1: chia)`, `2 (2: 10→5→2)`, `1 (3)`, `4 (3: 10→5→2→4)`, `8 (4)`. Các giá trị `20, 40, ...` đã vượt 13.
- Bảng giá của `2`: `2 (0)`, `4 (1: nhân)`, `8 (2)`, `1 (1: chia)`, `16 (3)`...
- Tìm cặp `(y1, y2)` với `y1 + y2 = 13`: `(5, 8)` chi phí `1 + 2 = 3`; `(10, 3)`? `3` không có trong bảng giá của `2` (từ `2` ta chỉ tạo ra các luỹ thừa của 2). `(4, 9)`, `(2, 11)`, `(1, 12)`? `9`, `11`, `12` không có trong bảng giá của phần tử còn lại. Chỉ có `(5, 8)` hợp lệ nên đáp án là `3`.

### 4.3. Tính tay ví dụ 2: `nums = [6, 3]`, `sum = 8`

`3 → 1 (1 phép) → 2 (2 phép)`. Dùng `6 + 2 = 8` với chi phí `0 + 2 = 2`. Nếu chỉ cho phép "toàn nhân rồi toàn chia" ta vẫn thấy `3 → 1 → 2`, nghĩa là chia trước, nhân sau. Điều đó gợi ý **dạng chuẩn** ở phần 6.

## 5. Cách 1 — Vét cạn: BFS bảng giá + thử mọi tổ hợp

**Ý tưởng "nguyên văn" (không dùng insight nào).**

1. Với mỗi `x`, chạy **BFS** trên đồ thị các giá trị: từ `v` đi tới `2v` và `v // 2`, mỗi cạnh tốn 1 phép. BFS cho chi phí nhỏ nhất để tới từng giá trị. Ta chặn giá trị trên một ngưỡng `cap` (đủ lớn, ví dụ `2 * (sum + max(nums))`) để BFS hữu hạn; `0` là giá trị vô ích nên không đi tiếp (nhưng vẫn có thể xuất hiện, không ảnh hưởng tổng).
2. Duyệt đệ quy qua các phần tử: bỏ qua hoặc chọn một giá trị trong bảng giá nếu còn vừa phần `rem` cần tạo.
3. Lấy chi phí nhỏ nhất khi `rem == 0`.

Cách này **đúng chắc chắn** theo đề nên ta dùng nó làm "trọng tài" để kiểm tra các cách sau.

In [6]:
from collections import deque

def price_table(x, cap):
    """BFS: dict {giá trị y: số phép ít nhất để biến x thành y}, chỉ giữ y <= cap."""
    dist = {x: 0}
    queue = deque([x])
    while queue:
        v = queue.popleft()
        for u in (2 * v, v // 2):
            if u <= cap and u not in dist:
                dist[u] = dist[v] + 1
                queue.append(u)
    return dist

def min_ops_bruteforce(nums, target):
    cap = 2 * (target + max(nums))
    tables = [price_table(x, cap) for x in nums]
    best = [10**9]

    def go(i, rem, cost):
        if rem == 0:
            best[0] = min(best[0], cost)
            return
        if i == len(nums):
            return
        go(i + 1, rem, cost)                       # không dùng phần tử i
        for y, c in tables[i].items():
            if 0 < y <= rem:
                go(i + 1, rem - y, cost + c)       # dùng phần tử i với giá trị y

    go(0, target, 0)
    return -1 if best[0] >= 10**9 else best[0]

print(sorted(price_table(6, 30).items()))
for nums, target, expected in EXAMPLES:
    print(nums, target, min_ops_bruteforce(nums, target), expected)

[(0, 3), (1, 2), (2, 3), (3, 1), (4, 4), (6, 0), (8, 5), (12, 1), (16, 6), (24, 2)]
[10, 2] 13 3 3
[6, 3] 8 2 2
[2, 2] 7 -1 -1


**Độ phức tạp của vét cạn.** Mỗi phần tử có tới `O(log cap)` giá trị trong bảng giá, nên số tổ hợp là khoảng `(log cap)^n` — **mũ theo `n`**. Với `n = 100` là không thể. Chỉ dùng được với `n ≤ 4`, giá trị nhỏ (như trong test ngẫu nhiên ở phần 12).

**Nút thắt:** ta đang duyệt *tổ hợp* trong khi chỉ cần biết "tổng `w` đã đạt được với chi phí tối thiểu bao nhiêu". Nhiều tổ hợp khác nhau cho cùng tổng `w` — ta nên gộp chúng lại: đó là quy hoạch động.

## 6. Bước đột phá

### 6.1. Dạng chuẩn: "chia `i` lần rồi nhân `j` lần"

**Insight 1.** Một phép nhân ngay sau đó (hoặc về sau) mà gặp một phép chia thì nó bị **triệt tiêu** vì `(2x) // 2 = x`. Nếu trong chuỗi có chỗ "nhân rồi chia" liền kề, ta xoá cả hai phép mà giá trị cuối không đổi, còn chi phí giảm 2. Lặp lại đến khi không còn cặp "nhân rồi chia" nào thì chuỗi có dạng **toàn chia trước, toàn nhân sau**.

Hệ quả: giá trị đạt được là

`y(i, j) = (x // 2^i) * 2^j`, chi phí `i + j`, với `i, j >= 0`.

(Chia `i` lần liên tiếp tương đương `x // 2^i`, vì `floor(floor(x/2)/2) = floor(x/4)`.)

### 6.2. Mỗi phần tử là một nhóm các lựa chọn

Với mỗi `x`, ta liệt kê mọi `(i, j)` sao cho `0 < y(i, j) <= sum`. Có tối đa `~ log2(x) * log2(sum)` lựa chọn, tức khoảng `9 * 13 = 117` với `x ≤ 500`, `sum ≤ 5000`. Nhiều `(i, j)` cho cùng `y` — ta chỉ giữ chi phí nhỏ nhất.

### 6.3. Quy tắc nếu… thì…

- Nếu `y(i, j) > sum` → bỏ (không bao giờ nằm trong tập con đúng tổng `sum`, vì tất cả giá trị đều dương).
- Nếu `y(i, j) = 0` → bỏ (không giúp gì, chỉ tốn phép).
- Nếu hai `(i, j)` khác nhau cho cùng `y` → giữ cái rẻ hơn.
- Mỗi phần tử gốc chọn **tối đa một** lựa chọn → knapsack theo nhóm.

**Bất biến DP.** Sau khi xử lý `k` phần tử đầu, `f[w]` = chi phí nhỏ nhất để chọn tập con của `k` phần tử đầu (mỗi phần tử một phiên bản) có tổng đúng `w`.

In [7]:
def options(x, S):
    """Bảng giá chuẩn của x: {y: chi phí nhỏ nhất}, chỉ giữ 0 < y <= S, với y = (x >> i) << j, chi phí i + j."""
    opt = {}
    i, base = 0, x
    while base > 0:                    # base = x // 2^i
        j, y = 0, base
        while y <= S:
            if y not in opt or opt[y] > i + j:
                opt[y] = i + j
            y *= 2
            j += 1
        base //= 2
        i += 1
    return opt

print(sorted(options(6, 30).items()))
# So với BFS: các giá trị <= 30 và chi phí phải khớp (BFS dùng cap lớn nên khớp phần <= 30)
bfs = {y: c for y, c in price_table(6, 120).items() if 0 < y <= 30}
print(bfs == options(6, 30))
for x in range(1, 40):
    a = {y: c for y, c in price_table(x, 400).items() if 0 < y <= 60}
    assert a == options(x, 60), x
print("Dạng chuẩn khớp BFS với mọi x < 40")

[(1, 2), (2, 3), (3, 1), (4, 4), (6, 0), (8, 5), (12, 1), (16, 6), (24, 2)]
True
Dạng chuẩn khớp BFS với mọi x < 40


## 7. Các cách tiếp theo

Từ bảng giá chuẩn, ta có 4 cách DP, từ dễ hiểu đến gọn nhất.

### 7.1. Cách 2 — DP 2 chiều `dp[k][w]` (nhóm knapsack, dễ hiểu nhất)

**Giải thích từng đoạn.**

- `dp[k][w]`: dùng `k` phần tử đầu, chi phí nhỏ nhất để có tổng đúng `w`.
- Khởi tạo: `dp[0][0] = 0`, còn lại `INF`.
- Chuyển trạng thái với phần tử thứ `k` (tức `nums[k-1]`):
  - bỏ qua: `dp[k][w] = dp[k-1][w]`;
  - chọn phiên bản `(y, c)`: `dp[k][w] = min(dp[k][w], dp[k-1][w-y] + c)`.
- Đáp án: `dp[n][sum]`.

Vì dùng hai hàng khác nhau nên **không** có vấn đề dùng lại phần tử.

In [8]:
def min_ops_dp2d(nums, target):
    INF = 10**9
    n = len(nums)
    dp = [[INF] * (target + 1) for _ in range(n + 1)]
    dp[0][0] = 0
    for k in range(1, n + 1):
        opts = options(nums[k - 1], target)
        for w in range(target + 1):
            best = dp[k - 1][w]                       # bỏ qua phần tử này
            for y, c in opts.items():
                if y <= w and dp[k - 1][w - y] + c < best:
                    best = dp[k - 1][w - y] + c
            dp[k][w] = best
    return -1 if dp[n][target] >= INF else dp[n][target]

for nums, target, expected in EXAMPLES:
    print(nums, target, min_ops_dp2d(nums, target), expected)

[10, 2] 13 3 3
[6, 3] 8 2 2
[2, 2] 7 -1 -1


**Độ phức tạp.** Với `n` phần tử, `S = sum`, mỗi phần tử có `L = O(log M · log S)` lựa chọn: Time `O(n · S · L)`, Space `O(n · S)`.
Muốn tiết kiệm bộ nhớ: mỗi hàng chỉ phụ thuộc hàng trước → chỉ cần giữ 2 hàng (cách 3).

### 7.2. Cách 3 — DP 1 chiều với mảng `g` mới cho mỗi phần tử

Chỉ cần hai mảng: `f` (trước khi xử lý phần tử) và `g` (sau). Với mỗi lựa chọn `(y, c)`: `g[w] = min(g[w], f[w-y] + c)`. Cuối phần tử, gán `f = g`. Space `O(S)`.

In [9]:
def min_ops_dp_rolling(nums, target):
    INF = 10**9
    f = [0] + [INF] * target
    for x in nums:
        g = f[:]                                  # mặc định: không dùng x
        for y, c in options(x, target).items():
            for w in range(y, target + 1):
                v = f[w - y] + c                   # đọc f cũ
                if v < g[w]:
                    g[w] = v
        f = g
    return -1 if f[target] >= INF else f[target]

for nums, target, expected in EXAMPLES:
    print(nums, target, min_ops_dp_rolling(nums, target), expected)

[10, 2] 13 3 3
[6, 3] 8 2 2
[2, 2] 7 -1 -1


### 7.3. Cách 4 — Top-down: đệ quy có nhớ `solve(k, rem)`

Hỏi: "dùng các phần tử `k, k+1, ..., n-1` để tạo tổng `rem`, tốn tối thiểu bao nhiêu phép?" Với phần tử `k` ta chọn: bỏ qua hoặc lấy một phiên bản `(y, c)` có `y <= rem`. Trả về `INF` nếu không thể; `solve(k, 0) = 0`.

Đây là cùng một DP viết theo chiều đệ quy. Số trạng thái `n · S`, mỗi trạng thái duyệt `L` lựa chọn. Chỉ các trạng thái *thực sự được gọi tới* mới được tính, nên đôi khi nhanh hơn với `S` lớn nhưng dễ chạm giới hạn đệ quy (độ sâu ≤ `n = 100` nên an toàn).

In [10]:
from functools import lru_cache

def min_ops_memo(nums, target):
    INF = 10**9
    n = len(nums)
    opts = [list(options(x, target).items()) for x in nums]

    @lru_cache(maxsize=None)
    def solve(k, rem):
        if rem == 0:
            return 0
        if k == n:
            return INF
        best = solve(k + 1, rem)                   # bỏ qua phần tử k
        for y, c in opts[k]:
            if y <= rem:
                best = min(best, solve(k + 1, rem - y) + c)
        return best

    ans = solve(0, target)
    return -1 if ans >= INF else ans

for nums, target, expected in EXAMPLES:
    print(nums, target, min_ops_memo(nums, target), expected)

[10, 2] 13 3 3
[6, 3] 8 2 2
[2, 2] 7 -1 -1


### 7.4. Cách 5 — DP 1 chiều duyệt `w` giảm dần (bản gọn để nộp)

Muốn bỏ luôn mảng phụ `g`, ta cập nhật **ngay trong `f`** nhưng duyệt `w` từ `S` xuống `0`. Khi xử lý `w`, mọi `f[w - y]` với `y >= 1` đều có chỉ số **nhỏ hơn** `w` nên chưa bị phần tử hiện tại đụng tới — chính là mẹo của knapsack 0-1 ở mục 3.3.

Trong code, với mỗi `w` ta duyệt:

1. `i = 0`: `y = x, 2x, 4x, ...` (chỉ nhân), chi phí `j`.
2. `i = 1, 2, ...`: `base = x >> i`, rồi `y = base, 2·base, ...`, chi phí `i + j`.

Các vòng `while` tự dừng khi `y > w`, nên không cần dict. Đây là dạng lời giải tham khảo của bài.

In [11]:
def min_ops_dp_inplace(nums, target):
    INF = 10**9
    f = [0] + [INF] * target
    for x in nums:
        for w in range(target, -1, -1):
            # 0 lần chia, chỉ nhân dần (biến i ở đây đếm số lần nhân)
            i, y = 0, x                              # nhánh không chia: y = x * 2^i, chi phí = số lần nhân
            while y <= w:
                f[w] = min(f[w], f[w - y] + i)
                i += 1
                y *= 2
            # chia i >= 1 lần rồi nhân j lần
            i, base = 1, x // 2
            while base > 0:
                j, z = 0, base
                while z <= w:
                    f[w] = min(f[w], f[w - z] + i + j)
                    j += 1
                    z *= 2
                i += 1
                base //= 2
    return -1 if f[target] >= INF else f[target]

for nums, target, expected in EXAMPLES:
    print(nums, target, min_ops_dp_inplace(nums, target), expected)

[10, 2] 13 3 3
[6, 3] 8 2 2
[2, 2] 7 -1 -1


**Lưu ý nhỏ.** Nếu để `y = 0` thì `while y <= w` lặp vô hạn (`0 * 2 = 0`). Code tránh được vì vòng ngoài chỉ chạy khi `base > 0` (đề đảm bảo `x >= 1`).

**Độ phức tạp.** `O(n · S · log M · log S)` thời gian, `O(S)` bộ nhớ. Số vòng thực tế nhỏ hơn đáng kể vì `y <= w` cắt sớm khi `w` nhỏ.

## 8. TRACE từng bước

Ta trace cách 3 (mảng `f` qua từng phần tử) trên ví dụ 2: `nums = [6, 3]`, `sum = 8`. Mảng `f` có chỉ số `0..8`.

| Sau phần tử | Bảng giá `{y: chi phí}` (y ≤ 8) | `f[0..8]` (INF viết là `.`) |
|---|---|---|
| khởi tạo | — | `0 . . . . . . . .` |
| `6` | `6:0, 3:1, 1:2, 2:3, 4:4, 8:5` | `0 2 3 1 4 . 0 . 5` |
| `3` | `3:0, 6:1, 1:1, 2:2, 4:3, ...` | `0 1 2 0 2 3 0 1 2` |

Đọc bảng: sau phần tử `6`, `f[6] = 0` (giữ nguyên 6) và `f[3] = 1` (chia 6 → 3). Sau phần tử `3`, `f[8] = 2`: lấy `6` (từ `f[6] = 0`) rồi thêm giá trị `2` của phần tử `3` (chi phí 2). Hàm trace dưới in từng cập nhật `g[w]` để bạn đối chiếu.

In [12]:
def trace_min_ops(nums, target):
    INF = 10**9
    show = lambda arr: " ".join("." if v >= INF else str(v) for v in arr)
    f = [0] + [INF] * target
    print("chỉ số :", " ".join(str(w) for w in range(target + 1)))
    print("khởi tạo:", show(f))
    for x in nums:
        opts = options(x, target)
        print(f"\nphần tử x={x}, bảng giá (y:chi phí) = {dict(sorted(opts.items()))}")
        g = f[:]
        for y, c in sorted(opts.items()):
            for w in range(y, target + 1):
                v = f[w - y] + c
                if v < g[w]:
                    print(f"   chọn y={y} (chi phí {c}): g[{w}] = f[{w - y}] + {c} = {v}")
                    g[w] = v
        f = g
        print("f sau x =", x, ":", show(f))
    ans = -1 if f[target] >= INF else f[target]
    print("\nĐáp án =", ans)
    return ans

trace_min_ops([6, 3], 8)

chỉ số : 0 1 2 3 4 5 6 7 8
khởi tạo: 0 . . . . . . . .

phần tử x=6, bảng giá (y:chi phí) = {1: 2, 2: 3, 3: 1, 4: 4, 6: 0, 8: 5}
   chọn y=1 (chi phí 2): g[1] = f[0] + 2 = 2
   chọn y=2 (chi phí 3): g[2] = f[0] + 3 = 3
   chọn y=3 (chi phí 1): g[3] = f[0] + 1 = 1
   chọn y=4 (chi phí 4): g[4] = f[0] + 4 = 4
   chọn y=6 (chi phí 0): g[6] = f[0] + 0 = 0
   chọn y=8 (chi phí 5): g[8] = f[0] + 5 = 5
f sau x = 6 : 0 2 3 1 4 . 0 . 5

phần tử x=3, bảng giá (y:chi phí) = {1: 1, 2: 2, 3: 0, 4: 3, 6: 1, 8: 4}
   chọn y=1 (chi phí 1): g[1] = f[0] + 1 = 1
   chọn y=1 (chi phí 1): g[4] = f[3] + 1 = 2
   chọn y=1 (chi phí 1): g[5] = f[4] + 1 = 5
   chọn y=1 (chi phí 1): g[7] = f[6] + 1 = 1
   chọn y=2 (chi phí 2): g[2] = f[0] + 2 = 2
   chọn y=2 (chi phí 2): g[5] = f[3] + 2 = 3
   chọn y=2 (chi phí 2): g[8] = f[6] + 2 = 2
   chọn y=3 (chi phí 0): g[3] = f[0] + 0 = 0
f sau x = 3 : 0 1 2 0 2 3 0 1 2

Đáp án = 2


2

### 8.2. Trace ví dụ 1 (`[10, 2]`, `sum = 13`)

Quan sát: sau phần tử `10`, `f[5] = 1` (chia một lần) và `f[10] = 0`. Sau phần tử `2`, ô `f[13]` được cập nhật từ `f[5] + 2 = 3` (lấy giá trị `8` của phần tử `2`) — đó là đáp án 3.

In [13]:
trace_min_ops([10, 2], 13)

chỉ số : 0 1 2 3 4 5 6 7 8 9 10 11 12 13
khởi tạo: 0 . . . . . . . . . . . . .

phần tử x=10, bảng giá (y:chi phí) = {1: 3, 2: 2, 4: 3, 5: 1, 8: 4, 10: 0}
   chọn y=1 (chi phí 3): g[1] = f[0] + 3 = 3
   chọn y=2 (chi phí 2): g[2] = f[0] + 2 = 2
   chọn y=4 (chi phí 3): g[4] = f[0] + 3 = 3
   chọn y=5 (chi phí 1): g[5] = f[0] + 1 = 1
   chọn y=8 (chi phí 4): g[8] = f[0] + 4 = 4
   chọn y=10 (chi phí 0): g[10] = f[0] + 0 = 0
f sau x = 10 : 0 3 2 . 3 1 . . 4 . 0 . . .

phần tử x=2, bảng giá (y:chi phí) = {1: 1, 2: 0, 4: 1, 8: 2}
   chọn y=1 (chi phí 1): g[1] = f[0] + 1 = 1
   chọn y=1 (chi phí 1): g[3] = f[2] + 1 = 3
   chọn y=1 (chi phí 1): g[6] = f[5] + 1 = 2
   chọn y=1 (chi phí 1): g[9] = f[8] + 1 = 5
   chọn y=1 (chi phí 1): g[11] = f[10] + 1 = 1
   chọn y=2 (chi phí 0): g[2] = f[0] + 0 = 0
   chọn y=2 (chi phí 0): g[4] = f[2] + 0 = 2
   chọn y=2 (chi phí 0): g[7] = f[5] + 0 = 1
   chọn y=2 (chi phí 0): g[12] = f[10] + 0 = 0
   chọn y=4 (chi phí 1): g[4] = f[0] + 1 = 1
   chọn y=4 (c

3

## 9. Code nộp LeetCode

Ta nộp **cách 3** (DP 1 chiều với mảng `g`) — an toàn (không lo thứ tự duyệt) và, theo phép đo ở phần 11, nhanh hơn cách 5 vì bảng giá được tính một lần mỗi phần tử rồi dedup. Lưu ý tham số tên `sum` đã che hàm dựng sẵn `sum`, nên trong code **không** gọi `sum(...)`.

In [14]:
class Solution:
    def minOperations(self, nums: list[int], sum: int) -> int:
        INF = 10**9
        f = [0] + [INF] * sum                 # f[w]: số phép ít nhất để một tập con có tổng đúng w

        for x in nums:
            # Bảng giá chuẩn của x: dạng (x >> i) << j, chi phí i + j, giữ 0 < y <= sum
            opts = {}
            i, base = 0, x
            while base > 0:
                j, y = 0, base
                while y <= sum:
                    if y not in opts or opts[y] > i + j:
                        opts[y] = i + j
                    y <<= 1
                    j += 1
                base >>= 1
                i += 1

            g = f[:]                          # mặc định: không dùng phần tử x
            for y, c in opts.items():
                for w in range(y, sum + 1):
                    v = f[w - y] + c          # luôn đọc f (cũ) -> mỗi phần tử tối đa 1 lần
                    if v < g[w]:
                        g[w] = v
            f = g

        return -1 if f[sum] >= INF else f[sum]


sol = Solution()
for nums, target, expected in EXAMPLES:
    assert sol.minOperations(nums, target) == expected, (nums, target)
print("3 ví dụ của đề: OK")

3 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) Dạng chuẩn đủ tổng quát.** Mọi chuỗi thao tác của một phần tử là một dãy `M` (nhân) và `D` (chia). Cặp `MD` liền kề cho cùng giá trị nên xoá nó không đổi kết quả, chỉ làm chi phí nhỏ đi. Lặp lại tới khi không còn `MD`: chuỗi còn lại có dạng `D...D M...M`. Do đó mọi giá trị đạt được, với chi phí nhỏ nhất, đều là `(x >> i) << j` với chi phí `i + j`. Ngược lại, mọi `(i, j)` đều là chuỗi hợp lệ. Vậy bảng giá chuẩn **vừa đủ**.

**(b) Giá trị dương, không vượt `sum`.** Tập con phải có tổng đúng `sum`; mọi số hạng là số nguyên **không âm**. Số hạng `0` không đóng góp, bỏ phần tử đó đỡ tốn phép. Số hạng `> sum` làm tổng vượt `sum` → không bao giờ dùng. Nên chỉ cần `0 < y <= sum`.

**(c) Tính đúng của DP.** Theo bất biến ở phần 6: sau xử lý phần tử thứ `k`, `f[w]` là chi phí nhỏ nhất trên mọi cách chọn tập con của `k` phần tử đầu với mỗi phần tử tối đa một phiên bản. Bước chuyển: phần tử `k` hoặc không dùng (giữ `f[w]`), hoặc dùng phiên bản `(y, c)` → ghép với tập con tốt nhất của `k - 1` phần tử đầu có tổng `w - y` (chính là `f` cũ). Vì dùng `f` cũ nên phần tử `k` không bị dùng hai lần.

**(d) Phần tử không dùng thì không tốn phép.** Ta chỉ tính chi phí cho phần tử nằm trong tập con. Đúng với đề, vì phép biến đổi chỉ cần thiết cho phần tử được chọn.

## 11. Phân tích Time/Space

Đặt `n = len(nums)`, `M = max(nums)` (≤ 500), `S = sum` (≤ 5000).

| Phần | Chi phí |
|---|---|
| Bảng giá mỗi phần tử | `O(log M · log S)` lựa chọn (tối đa khoảng 9 × 13) |
| DP mỗi phần tử | `O(S · số lựa chọn)` |
| **Tổng thời gian** | `O(n · S · log M · log S)` |
| **Bộ nhớ** | `O(S)` cho `f` và `g` (cộng `O(log M · log S)` cho bảng giá) |

Ước lượng cận trên: `100 × 5000 × ~100 ≈ 5·10^7` phép cập nhật — vẫn ổn trong C++/Java; với Python, ta giảm hằng số bằng dedup và cắt `y <= S`. Ta đo nhanh bên dưới (con số tuyệt đối tuỳ máy chạy notebook).

In [15]:
import random, time

rng = random.Random(4041)
big_nums = [500] * 100
for name, fn in [("rolling dp", min_ops_dp_rolling), ("inplace dp", min_ops_dp_inplace), ("Solution", Solution().minOperations)]:
    t = time.time()
    r = fn(big_nums, 4999)   # chỉ có bội của 500, nên cần nhiều phép chia/nhân
    print(f"{name:11s} n=100, sum=4999 -> {r}  ({time.time() - t:.2f}s)")

rolling dp  n=100, sum=4999 -> 7  (0.89s)


inplace dp  n=100, sum=4999 -> 7  (2.37s)


Solution    n=100, sum=4999 -> 7  (0.98s)


## 12. Bộ test

Ta kiểm tra: ví dụ đề, edge case, rồi **đối chiếu mọi cách** với vét cạn trên input ngẫu nhiên nhỏ (seed cố định).

In [16]:
ALL = [min_ops_dp2d, min_ops_dp_rolling, min_ops_memo, min_ops_dp_inplace, Solution().minOperations]

EDGE = [
    ([1], 1, 0),                 # đã đủ
    ([1], 2, 1),                 # 1 -> 2
    ([1], 3, -1),                # từ 1 chỉ ra được các luỹ thừa của 2
    ([5], 5, 0),
    ([5], 10, 1),                # 5 -> 10
    ([5], 2, 1),                 # 5 // 2 = 2: một phép chia
    ([3, 3], 6, 0),
    ([3, 3], 5, 2),              # 3 + 2, với 3 -> 1 -> 2 tốn 2 phép
    ([1, 1, 1], 7, 3),           # 4 + 2 + 1: 1->2->4 (2 phép) + 1->2 (1 phép) + 1 (0 phép)
    ([500], 5000, -1),           # 500 * 2^j chỉ ra 500, 1000, 2000, 4000
    ([500, 500], 1500, 1),       # 500 + 1000
]
for nums, target, expected in EDGE:
    for fn in ALL:
        assert fn(nums, target) == expected, (fn.__name__, nums, target, fn(nums, target), expected)
    if len(nums) <= 3 and target <= 200:
        assert min_ops_bruteforce(nums, target) == expected, (nums, target)
print("Edge case: mọi cách đúng (và khớp vét cạn khi vét cạn kịp)")

for fn in ALL:
    for nums, target, expected in EXAMPLES:
        assert fn(nums, target) == expected, (fn.__name__, nums, target)
print("Ví dụ đề: tất cả cách đều đúng")

Edge case: mọi cách đúng (và khớp vét cạn khi vét cạn kịp)
Ví dụ đề: tất cả cách đều đúng


In [17]:
rng = random.Random(404101)
for _ in range(500):
    n = rng.randint(1, 4)
    nums = [rng.randint(1, 14) for _ in range(n)]
    target = rng.randint(1, 40)
    ref = min_ops_bruteforce(nums, target)
    for fn in ALL:
        got = fn(nums, target)
        assert got == ref, (fn.__name__, nums, target, got, ref)
print("500 input ngẫu nhiên nhỏ: mọi cách khớp vét cạn")

# Đối chiếu các DP với nhau trên input lớn hơn (không vét cạn được)
rng = random.Random(7)
for _ in range(60):
    n = rng.randint(1, 12)
    nums = [rng.randint(1, 500) for _ in range(n)]
    target = rng.randint(1, 600)
    res = {fn.__name__: fn(nums, target) for fn in ALL}
    assert len(set(res.values())) == 1, (nums, target, res)
print("60 input lớn hơn: các DP khớp nhau")

500 input ngẫu nhiên nhỏ: mọi cách khớp vét cạn


60 input lớn hơn: các DP khớp nhau


In [18]:
# Bất biến: input không bị thay đổi, kết quả -1 khi tổng không thể
nums = [2, 2]
snapshot = nums[:]
assert Solution().minOperations(nums, 7) == -1 and nums == snapshot
# Phản ví dụ cho "chỉ nhân": [6,3], 8 cần chia (3 -> 1) rồi nhân (1 -> 2)
def only_multiply(nums, target):
    # SAI: chỉ cho phép nhân, bỏ qua chia
    INF = 10**9
    f = [0] + [INF] * target
    for x in nums:
        g = f[:]
        y, c = x, 0
        while y <= target:
            for w in range(y, target + 1):
                g[w] = min(g[w], f[w - y] + c)
            y, c = y * 2, c + 1
        f = g
    return -1 if f[target] >= INF else f[target]
print("only_multiply([6,3],8) =", only_multiply([6, 3], 8), "(sai) vs đúng =", Solution().minOperations([6, 3], 8))

only_multiply([6,3],8) = -1 (sai) vs đúng = 2


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Duyệt `w` từ nhỏ lên khi cập nhật tại chỗ

**Code sai:**
```python
for x in nums:
    for w in range(y, S + 1):       # tăng dần
        f[w] = min(f[w], f[w - y] + c)
```
**Vì sao sai:** `f[w - y]` có thể đã bị chính phần tử `x` cập nhật → ta dùng `x` hai lần (như cell 3.3).
**Code đúng:** duyệt giảm dần, hoặc dùng mảng `g` mới như bản nộp.

In [19]:
def wrong_ascending(nums, target):
    INF = 10**9
    f = [0] + [INF] * target
    for x in nums:
        for y, c in options(x, target).items():
            for w in range(y, target + 1):         # SAI: tăng dần, ghi đè f ngay
                f[w] = min(f[w], f[w - y] + c)
    return -1 if f[target] >= INF else f[target]

print("sai:", wrong_ascending([5], 10), " đúng:", Solution().minOperations([5], 10))
rng = random.Random(1)
for _ in range(2000):
    nums = [rng.randint(1, 6) for _ in range(rng.randint(1, 3))]
    t = rng.randint(1, 20)
    if wrong_ascending(nums, t) != Solution().minOperations(nums, t):
        print("phản ví dụ:", nums, t, wrong_ascending(nums, t), Solution().minOperations(nums, t)); break

sai: 0  đúng: 1
phản ví dụ: [5] 3 3 -1


### Lỗi 2 — Chỉ xét "nhân" hoặc chỉ xét "chia" (bỏ qua đan xen)

**Code sai:** coi mỗi phần tử chỉ có `x * 2^j` hoặc `x // 2^i`.
**Vì sao sai:** ví dụ 2 cần `3 → 1 → 2` (chia rồi nhân). Xem `only_multiply` ở phần 12, trả về `-1` thay vì `2`.
**Code đúng:** bảng giá chuẩn `(x >> i) << j` với cả hai `i, j`.

### Lỗi 3 — Đếm cả phép biến đổi cho phần tử không được chọn / quên `y = 0`

**Code sai:** cộng chi phí của mọi phần tử, hoặc để vòng `while` chạy với `y = 0`.
**Vì sao sai:** (a) chỉ phần tử trong tập con mới cần biến đổi; (b) `0 * 2 = 0` làm vòng lặp vô hạn.
**Code đúng:** vòng `while base > 0` và `y <= sum`; phần tử không chọn giữ `g = f[:]`.

### Lỗi 4 — Khởi tạo `f` toàn số 0

**Code sai:** `f = [0] * (sum + 1)`.
**Vì sao sai:** `f[w] = 0` nghĩa là "tổng `w` đạt được mà không tốn phép nào" — sai với mọi `w > 0`. Chỉ `f[0] = 0` mới đúng; các tổng còn lại phải là `INF` ("chưa đạt được").
**Code đúng:** `f = [0] + [INF] * sum`.

In [20]:
def wrong_init(nums, target):
    INF = 10**9
    f = [0] * (target + 1)                 # SAI
    for x in nums:
        g = f[:]
        for y, c in options(x, target).items():
            for w in range(y, target + 1):
                g[w] = min(g[w], f[w - y] + c)
        f = g
    return -1 if f[target] >= INF else f[target]

print("wrong_init([2,2],7) =", wrong_init([2, 2], 7), "(đúng: -1)")
print("wrong_init([6,3],8) =", wrong_init([6, 3], 8), "(đúng: 2)")

wrong_init([2,2],7) = 0 (đúng: -1)
wrong_init([6,3],8) = 0 (đúng: 2)


## 14. Mẹo & nhận dạng pattern

- **"Mỗi phần tử có nhiều biến thể với chi phí khác nhau, chọn một"** → knapsack theo nhóm.
- **"Thao tác có thể triệt tiêu nhau"** (nhân rồi chia, +1 rồi -1, mở rồi đóng) → tìm **dạng chuẩn** bằng cách xoá cặp triệt tiêu, thường làm số trạng thái giảm mạnh.
- **Tổng đích nhỏ (≤ 5000) và `n` nhỏ (≤ 100)** → DP theo giá trị tổng là hợp lý (pseudo-polynomial).
- Nhìn `sum ≤ 5000` và `nums[i] ≤ 500` → gợi ý `O(n · S · log)`.
- Khi so sánh DP với vét cạn, đặt **ngưỡng `cap` đủ lớn** cho BFS để khỏi bỏ sót đường đi "nhân quá `sum` rồi chia về".
- Luôn tách "giá trị đạt được" và "chi phí": đó là bảng giá.

## 15. Biến thể

1. **Chi phí khác nhau cho nhân và chia:** chia tốn `a`, nhân tốn `b` → chi phí `a·i + b·j`. Dạng chuẩn vẫn đúng (xoá cặp "nhân rồi chia" luôn tiết kiệm `a + b > 0`), DP y nguyên. Đây là Bài 5.
2. **Mỗi phần tử dùng nhiều lần** (knapsack không giới hạn): duyệt `w` tăng dần khi cập nhật tại chỗ.
3. **Tổng ít nhất `sum`:** lấy `min(f[w] for w >= sum)` với `f` mở rộng tới `sum + max giá trị`.
4. **Chỉ cần xác định có/không:** bỏ chi phí, dùng bitmask `mask |= mask << y`.
5. **Giới hạn tổng số phép ≤ K:** trả về `f[sum] <= K` hoặc thêm chiều `k` vào DP.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn BFS + tổ hợp | Bảng giá BFS, thử mọi tổ hợp | mũ theo `n` | `O(n·log)` | Trọng tài kiểm tra, `n ≤ 4` |
| 2. DP 2 chiều | `dp[k][w]` knapsack nhóm | `O(n·S·L)` | `O(n·S)` | Học, dễ hiểu |
| 3. DP 1 chiều + `g` | Hai mảng, đọc `f` cũ | `O(n·S·L)` | `O(S)` | **Bản nộp** |
| 4. Top-down memo | `solve(k, rem)` | `O(n·S·L)` (chỉ trạng thái dùng tới) | `O(n·S)` | Khi muốn viết đệ quy |
| 5. DP tại chỗ giảm dần | Duyệt `w` giảm, vòng `while` | `O(n·S·log M·log S)` | `O(S)` | Code gọn, hằng số khác |

`L` = số lựa chọn mỗi phần tử ≈ `O(log M · log S)`.

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `options(5, 20)` trả về dict gì? (Dict `{giá trị y: chi phí nhỏ nhất}` với `0 < y <= 20`.)

**Bài 2 (tính tay).** `nums = [4]`, `sum = 6`. Liệt kê các giá trị ≤ 6 mà phần tử `4` biến thành được, rồi cho đáp án (hoặc `-1`).

**Bài 3 (điền TODO).** Hoàn thiện `build_options(x, S)` trả về dict `{y: chi phí nhỏ nhất}` theo dạng chuẩn.

**Bài 4 (tự viết lại).** Viết `my_min_ops(nums, target)` dùng DP 1 chiều với mảng phụ `g`.

**Bài 5 (biến thể / mini challenge).** Chia tốn `a` điểm, nhân tốn `b` điểm (mỗi phép). Viết `min_cost_weighted(nums, target, a, b)` trả về tổng chi phí nhỏ nhất.

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def build_options(x, S):
    # TODO 1: tạo dict opts rỗng
    # TODO 2: vòng ngoài: i = số lần chia, base = x >> i, dừng khi base == 0
    # TODO 3: vòng trong: j = số lần nhân, y = base << j, dừng khi y > S
    # TODO 4: nếu y chưa có trong opts hoặc i + j nhỏ hơn -> cập nhật
    pass

# Bài 4 — TODO
def my_min_ops(nums, target):
    # TODO: f = [0] + [INF]*target; với mỗi x: g = f[:]; dùng build_options; f = g
    pass

# Bài 5 — TODO
def min_cost_weighted(nums, target, a, b):
    # TODO: giống bài 4 nhưng chi phí của (i chia, j nhân) là a*i + b*j
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: chia dần `5 → 2 → 1 → 0`; dừng khi gặp `0`.
- Mức 2: từ mỗi giá trị đó nhân lên tới khi vượt 20, chi phí = số lần chia + số lần nhân.
- Mức 3: từ `5`: `5, 10, 20`; từ `2` (1 phép chia): `2, 4, 8, 16`; từ `1` (2 phép chia): `1, 2, 4, 8, 16` — nhưng `2, 4, 8, 16` đã có chi phí nhỏ hơn nên chỉ thêm `1`.

**Bài 2.**
- Mức 1: từ `4` chia được `2`, `1`; nhân được `8, 16, ...` (vượt 6).
- Mức 2: giá trị ≤ 6 gồm `4 (0 phép)`, `2 (1)`, `1 (2)`.
- Mức 3: chỉ có một phần tử nên tổng = giá trị của nó; `6` không nằm trong danh sách → `-1`.

**Bài 3.**
- Mức 1: hai vòng `while` lồng nhau.
- Mức 2: vòng ngoài `base = x`, cuối mỗi vòng `base //= 2`, `i += 1`.
- Mức 3: vòng trong `y = base; j = 0; while y <= S: ...; y *= 2; j += 1`.

**Bài 4.**
- Mức 1: `INF = 10**9`.
- Mức 2: với mỗi `(y, c)`, `for w in range(y, target + 1)`.
- Mức 3: đọc `f` cũ, ghi vào `g`, cuối phần tử `f = g`.

**Bài 5.**
- Mức 1: chỉ phần tính chi phí thay đổi.
- Mức 2: dạng chuẩn vẫn đúng với `a, b > 0`.
- Mức 3: thay `i + j` bằng `a * i + b * j` trong bảng giá.

## 19. Đáp án tham khảo

In [22]:
# Bài 1
assert options(5, 20) == {5: 0, 10: 1, 20: 2, 2: 1, 4: 2, 8: 3, 16: 4, 1: 2}
# Bài 2
assert options(4, 6) == {4: 0, 2: 1, 1: 2}
assert Solution().minOperations([4], 6) == -1

# Bài 3
def build_options_ans(x, S):
    opts = {}
    i, base = 0, x
    while base > 0:
        j, y = 0, base
        while y <= S:
            if y not in opts or opts[y] > i + j:
                opts[y] = i + j
            y *= 2
            j += 1
        base //= 2
        i += 1
    return opts

for x in range(1, 100):
    assert build_options_ans(x, 200) == options(x, 200)
print("Bài 1–3 OK")

Bài 1–3 OK


In [23]:
# Bài 4
def my_min_ops_ans(nums, target):
    INF = 10**9
    f = [0] + [INF] * target
    for x in nums:
        g = f[:]
        for y, c in build_options_ans(x, target).items():
            for w in range(y, target + 1):
                if f[w - y] + c < g[w]:
                    g[w] = f[w - y] + c
        f = g
    return -1 if f[target] >= INF else f[target]

# Bài 5
import heapq

def min_cost_weighted_ans(nums, target, a, b):
    INF = 10**9
    f = [0] + [INF] * target
    for x in nums:
        opts = {}
        i, base = 0, x
        while base > 0:
            j, y = 0, base
            while y <= target:
                cost = a * i + b * j
                if y not in opts or opts[y] > cost:
                    opts[y] = cost
                y *= 2
                j += 1
            base //= 2
            i += 1
        g = f[:]
        for y, c in opts.items():
            for w in range(y, target + 1):
                if f[w - y] + c < g[w]:
                    g[w] = f[w - y] + c
        f = g
    return -1 if f[target] >= INF else f[target]

def dijkstra_table(x, cap, a, b):
    # Trọng tài cho bài 5: Dijkstra trên đồ thị giá trị (chia tốn a, nhân tốn b)
    dist = {x: 0}
    pq = [(0, x)]
    while pq:
        d, v = heapq.heappop(pq)
        if d > dist[v]:
            continue
        for u, w in ((2 * v, b), (v // 2, a)):
            if u <= cap and d + w < dist.get(u, 10**9):
                dist[u] = d + w
                heapq.heappush(pq, (d + w, u))
    return dist

def weighted_bruteforce(nums, target, a, b):
    cap = 2 * (target + max(nums))
    tables = [dijkstra_table(x, cap, a, b) for x in nums]
    best = [10**9]
    def go(i, rem, cost):
        if rem == 0:
            best[0] = min(best[0], cost)
            return
        if i == len(nums):
            return
        go(i + 1, rem, cost)
        for y, c in tables[i].items():
            if 0 < y <= rem:
                go(i + 1, rem - y, cost + c)
    go(0, target, 0)
    return -1 if best[0] >= 10**9 else best[0]

rng = random.Random(99)
for _ in range(200):
    nums = [rng.randint(1, 10) for _ in range(rng.randint(1, 3))]
    t = rng.randint(1, 30)
    assert my_min_ops_ans(nums, t) == Solution().minOperations(nums, t)
    a, b = rng.randint(1, 4), rng.randint(1, 4)
    assert min_cost_weighted_ans(nums, t, a, b) == weighted_bruteforce(nums, t, a, b), (nums, t, a, b)
assert min_cost_weighted_ans([10, 2], 13, 1, 1) == 3
assert min_cost_weighted_ans([10, 2], 13, 5, 1) == 7      # chia 1 lần (5) + nhân 2 lần (2)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template knapsack theo nhóm, bảng giá chuẩn:**

```python
INF = 10**9
f = [0] + [INF] * S
for x in nums:
    opts = {}
    i, base = 0, x
    while base > 0:                       # chia i lần
        j, y = 0, base
        while y <= S:                     # nhân j lần
            opts[y] = min(opts.get(y, INF), i + j)
            y *= 2; j += 1
        base //= 2; i += 1
    g = f[:]                              # mặc định: không dùng x
    for y, c in opts.items():
        for w in range(y, S + 1):
            g[w] = min(g[w], f[w - y] + c)   # đọc f cũ
    f = g
return -1 if f[S] >= INF else f[S]
```

**Checklist trước khi submit:**

- [ ] Rút gọn chuỗi thao tác về dạng "chia `i` lần rồi nhân `j` lần".
- [ ] Bỏ giá trị `0` và giá trị `> sum`.
- [ ] Mỗi phần tử chọn tối đa **một** phiên bản (đọc `f` cũ hoặc duyệt `w` giảm).
- [ ] `f[0] = 0`, còn lại `INF = 10**9`; kết thúc kiểm tra `>= INF`.
- [ ] Không gọi hàm `sum()` bên trong vì tham số cùng tên.
- [ ] Đối chiếu với vét cạn trên input nhỏ trước khi nộp.